In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:

# Step 2: Data ingestion

appointments = pd.read_csv('/content/appointments.csv.csv')
clinics = pd.read_csv('/content/clinics.csv.csv')
followup_notes = pd.read_csv('/content/followup_notes.csv.csv')

print("All datasets loaded successfully!")

print("Appointments:", appointments.shape)
print("Clinics:", clinics.shape)
print("Follow-up Notes:", followup_notes.shape)


All datasets loaded successfully!
Appointments: (1020, 14)
Clinics: (8, 6)
Follow-up Notes: (30, 3)


In [ ]:

# Step 3: Initial Data Quality Audit

datasets = {
    "Appointments": appointments,
    "Clinics": clinics,
    "Follow-up Notes": followup_notes
}

for name, df in datasets.items():
    print("=" * 50)
    print(f"DATA QUALITY AUDIT: {name}")
    print("=" * 50)

    print("Total Rows:", df.shape[0])
    print("Total Columns:", df.shape[1])

    print("\nColumn Names:")
    print(df.columns.tolist())

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nData Types:")
    print(df.dtypes)

    print("\n")


DATA QUALITY AUDIT: Appointments
Total Rows: 1020
Total Columns: 14

Column Names:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']

Missing Values:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64

Duplicate Rows:
20

Data Types:
appointment_id      object
patient_id          object
clinic_id           object
booked_on           object
appt_date           object
age                float64
gender              object
distance_km        float64
fee_inr              int64
booking_channel     object
visit_type          object
reminder_sent       object
prior_no_shows       int64
st

In [ ]:

# Step 4.1: Duplicate Removal

# Preserve the original datasets
appointments_clean = appointments.copy()
clinics_clean = clinics.copy()
notes_clean = followup_notes.copy()

# Count records before cleaning
before = len(appointments_clean)

# Remove exact duplicate records
appointments_clean = appointments_clean.drop_duplicates()

# Reset row index
appointments_clean = appointments_clean.reset_index(drop=True)

# Validate cleaning results
after = len(appointments_clean)

print("DUPLICATE REMOVAL REPORT")
print("-" * 40)
print("Original records:", before)
print("Duplicates removed:", before - after)
print("Remaining records:", after)
print("Remaining duplicates:", appointments_clean.duplicated().sum())

# Check for repeated appointment IDs
print("Repeated appointment IDs:",
      appointments_clean["appointment_id"].duplicated().sum())


DUPLICATE REMOVAL REPORT
----------------------------------------
Original records: 1020
Duplicates removed: 20
Remaining records: 1000
Remaining duplicates: 0
Repeated appointment IDs: 0


In [ ]:

# Step 4.2: Missing Value Assessment

print("MISSING VALUE REPORT")
print("-" * 40)

missing_report = pd.DataFrame({
    "Missing_Count": appointments_clean.isnull().sum(),
    "Missing_Percentage": (
        appointments_clean.isnull().mean() * 100
    ).round(2)
})

display(missing_report[
    missing_report["Missing_Count"] > 0
])

print("\nNUMERICAL DATA SUMMARY")
display(appointments_clean[
    ["age", "distance_km", "fee_inr", "prior_no_shows"]
].describe())

print("\nAPPOINTMENT STATUS DISTRIBUTION")
print(appointments_clean["status"].value_counts(dropna=False))

print("\nFOLLOW-UP NOTES")
print("Missing notes:", notes_clean["note_text"].isnull().sum())


MISSING VALUE REPORT
----------------------------------------


,Missing_Count,Missing_Percentage
appt_date,1,0.1
age,12,1.2
distance_km,20,2.0
status,4,0.4



NUMERICAL DATA SUMMARY


,age,distance_km,fee_inr,prior_no_shows
count,988.000000,980.000000,1000.000000,1000.000000
mean,48.526316,10.015510,648.900000,0.632000
std,18.996622,11.089426,343.972538,0.947351
min,-4.000000,0.600000,300.000000,0.000000
25%,33.000000,4.600000,500.000000,0.000000
50%,48.000000,8.200000,650.000000,0.000000
75%,63.000000,13.100000,750.000000,1.000000
max,250.000000,221.000000,7500.000000,5.000000



APPOINTMENT STATUS DISTRIBUTION
status
Attended    777
No-Show     219
NaN           4
Name: count, dtype: int64

FOLLOW-UP NOTES
Missing notes: 1


In [ ]:

# Step 4.3: Numerical Data Validation

# Identify invalid age values
invalid_age = (
    appointments_clean["age"].notna()
    & ~appointments_clean["age"].between(0, 120)
)

print("Invalid age records:", invalid_age.sum())

# Replace invalid ages with missing values
appointments_clean.loc[invalid_age, "age"] = np.nan

# Check unusual distance and fee values
print("\nEXTREME VALUE REVIEW")

for col in ["distance_km", "fee_inr"]:

    q1 = appointments_clean[col].quantile(0.25)
    q3 = appointments_clean[col].quantile(0.75)

    iqr = q3 - q1
    upper_limit = q3 + 1.5 * iqr

    outliers = appointments_clean[col] > upper_limit

    print("\nColumn:", col)
    print("Upper IQR limit:", round(upper_limit, 2))
    print("Potential outliers:", outliers.sum())

# Validate the corrected age column
print("\nAGE VALIDATION REPORT")
print("Minimum age:", appointments_clean["age"].min())
print("Maximum age:", appointments_clean["age"].max())
print("Missing ages:", appointments_clean["age"].isna().sum())

print("\nNumerical validation completed!")


Invalid age records: 4

EXTREME VALUE REVIEW

Column: distance_km
Upper IQR limit: 25.85
Potential outliers: 26

Column: fee_inr
Upper IQR limit: 1125.0
Potential outliers: 3

AGE VALIDATION REPORT
Minimum age: 0.0
Maximum age: 78.0
Missing ages: 16

Numerical validation completed!


In [ ]:

# Step 4.4: Missing Value Treatment

# Create missing value indicators
appointments_clean["age_missing"] = (
    appointments_clean["age"].isna().astype(int)
)

appointments_clean["distance_missing"] = (
    appointments_clean["distance_km"].isna().astype(int)
)

appointments_clean["date_missing"] = (
    appointments_clean["appt_date"].isna().astype(int)
)

appointments_clean["status_missing"] = (
    appointments_clean["status"].isna().astype(int)
)

# Identify unavailable follow-up notes
notes_clean["note_missing"] = (
    notes_clean["note_text"].isna().astype(int)
)

# Generate missing value treatment report
print("MISSING VALUE TREATMENT REPORT")
print("-" * 40)

for col in ["age_missing", "distance_missing",
            "date_missing", "status_missing"]:
    print(col, ":", appointments_clean[col].sum())

print("Missing follow-up notes:",
      notes_clean["note_missing"].sum())

print("\nTotal appointment records:",
      len(appointments_clean))

print("\nMissing value indicators created successfully!")


MISSING VALUE TREATMENT REPORT
----------------------------------------
age_missing : 16
distance_missing : 20
date_missing : 1
status_missing : 4
Missing follow-up notes: 1

Total appointment records: 1000

Missing value indicators created successfully!


In [ ]:

# Step 4.5: Date Standardization

# Convert text into datetime format
for col in ["booked_on", "appt_date"]:
    appointments_clean[col] = pd.to_datetime(
        appointments_clean[col],
        format="mixed",
        errors="coerce"
    )

# Calculate appointment lead time in days
appointments_clean["lead_time_days"] = (
    appointments_clean["appt_date"].dt.normalize()
    - appointments_clean["booked_on"].dt.normalize()
).dt.days

# Check negative lead times
negative_lead = appointments_clean["lead_time_days"] < 0

# Update missing date indicator
appointments_clean["date_missing"] = (
    appointments_clean["appt_date"].isna().astype(int)
)

# Generate validation report
print("DATE STANDARDIZATION REPORT")
print("-" * 40)

print("Missing or invalid booking dates:",
      appointments_clean["booked_on"].isna().sum())

print("Missing or invalid appointment dates:",
      appointments_clean["appt_date"].isna().sum())

print("Negative lead times:", negative_lead.sum())

print("\nLEAD TIME SUMMARY")
print(appointments_clean["lead_time_days"].describe())

print("\nDate standardization completed!")


DATE STANDARDIZATION REPORT
----------------------------------------
Missing or invalid booking dates: 0
Missing or invalid appointment dates: 6
Negative lead times: 0

LEAD TIME SUMMARY
count    994.000000
mean      11.830986
std        8.756273
min        1.000000
25%        5.000000
50%       10.000000
75%       16.000000
max       45.000000
Name: lead_time_days, dtype: float64

Date standardization completed!


In [ ]:

# Step 4.6: Categorical Data Audit

categorical_columns = [
    "gender",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "status"
]

print("CATEGORICAL DATA AUDIT")
print("=" * 40)

for col in categorical_columns:
    print("\nColumn:", col)
    print("-" * 30)
    print(appointments_clean[col].value_counts(
        dropna=False
    ).to_string())

print("\nCLINIC CATEGORY AUDIT")
print("=" * 40)

for col in ["city", "tier"]:
    print("\nColumn:", col)
    print(clinics_clean[col].value_counts(
        dropna=False
    ).to_string())

print("\nCategorical audit completed!")


CATEGORICAL DATA AUDIT

Column: gender
------------------------------
gender
F    562
M    438

Column: booking_channel
------------------------------
booking_channel
App         358
Phone       342
Walk-in     230
APP          10
Walk-in      10
app           9
PHONE         7
 walk-in      5
App           5
Walk-In       5
phone         5
 App          5
 phone        4
WALK-IN       3
Phone         1
walk-in       1

Column: visit_type
------------------------------
visit_type
Consultation    593
Follow-up       407

Column: reminder_sent
------------------------------
reminder_sent
Yes    568
No     432

Column: status
------------------------------
status
Attended    777
No-Show     219
NaN           4

CLINIC CATEGORY AUDIT

Column: city
city
Pune       2
Jaipur     2
Nagpur     1
Kochi      1
Indore     1
Lucknow    1

Column: tier
tier
Tier-2    6
Metro     2

Categorical audit completed!


In [ ]:

# Step 4.7: Categorical Standardization

# Count categories before cleaning
before_categories = appointments_clean[
    "booking_channel"
].nunique()

# Standardize booking channel
appointments_clean["booking_channel"] = (
    appointments_clean["booking_channel"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "app": "App",
        "phone": "Phone",
        "walk-in": "Walk-in"
    })
)

# Remove unnecessary spaces from other categories
for col in ["gender", "visit_type",
            "reminder_sent", "status"]:
    appointments_clean[col] = (
        appointments_clean[col]
        .astype("string")
        .str.strip()
    )

# Count categories after cleaning
after_categories = appointments_clean[
    "booking_channel"
].nunique()

# Validation report
print("CATEGORICAL CLEANING REPORT")
print("=" * 40)

print("Categories before:", before_categories)
print("Categories after:", after_categories)

print("\nSTANDARDIZED BOOKING CHANNELS")
print(appointments_clean["booking_channel"].value_counts())

print("\nGENDER DISTRIBUTION")
print(appointments_clean["gender"].value_counts(dropna=False))

print("\nTotal appointments:", len(appointments_clean))

print("\nCategorical standardization completed!")


CATEGORICAL CLEANING REPORT
Categories before: 16
Categories after: 3

STANDARDIZED BOOKING CHANNELS
booking_channel
App        387
Phone      359
Walk-in    254
Name: count, dtype: Int64

GENDER DISTRIBUTION
gender
F    562
M    438
Name: count, dtype: Int64

Total appointments: 1000

Categorical standardization completed!


In [ ]:

# Step 4.8: Data Integrity Validation

# Recover original appointment dates for inspection
original_dates = (
    appointments.drop_duplicates()
    .set_index("appointment_id")["appt_date"]
)

# Identify missing or unparsed dates
invalid_dates = appointments_clean[
    appointments_clean["appt_date"].isna()
]

print("DATE ERROR REVIEW")
print("=" * 40)

print("Original unparsed date values:")
print(
    invalid_dates["appointment_id"]
    .map(original_dates)
    .value_counts(dropna=False)
)

# Check primary keys
print("\nPRIMARY KEY VALIDATION")
print("=" * 40)

print("Repeated clinic IDs:",
      clinics_clean["clinic_id"].duplicated().sum())

print("Repeated note IDs:",
      notes_clean["note_id"].duplicated().sum())

# Check relationships between datasets
print("\nFOREIGN KEY VALIDATION")
print("=" * 40)

unmatched_clinics = (
    ~appointments_clean["clinic_id"]
    .isin(clinics_clean["clinic_id"])
).sum()

unmatched_notes = (
    ~notes_clean["appointment_id"]
    .isin(appointments_clean["appointment_id"])
).sum()

print("Unmatched clinic references:",
      unmatched_clinics)

print("Unmatched appointment references:",
      unmatched_notes)

print("\nData integrity validation completed!")


DATE ERROR REVIEW
Original unparsed date values:
appointment_id
not booked    1
2025-06-31    1
2025-04-31    1
TBD           1
NaN           1
2025-02-30    1
Name: count, dtype: int64

PRIMARY KEY VALIDATION
Repeated clinic IDs: 0
Repeated note IDs: 0

FOREIGN KEY VALIDATION
Unmatched clinic references: 0
Unmatched appointment references: 0

Data integrity validation completed!


In [ ]:

# Step 4.9: Data Integration

# Merge appointments with clinic information
master_df = appointments_clean.merge(
    clinics_clean,
    on="clinic_id",
    how="left",
    validate="many_to_one"
)

# Validate the integrated dataset
print("DATA INTEGRATION REPORT")
print("=" * 40)

print("Appointment records:", len(appointments_clean))
print("Clinic records:", len(clinics_clean))
print("Integrated records:", len(master_df))
print("Total columns:", master_df.shape[1])

print("Duplicate appointment IDs:",
      master_df["appointment_id"].duplicated().sum())

print("Unmatched clinics:",
      master_df["clinic_name"].isna().sum())

# Confirm that no appointments were lost
assert len(master_df) == len(appointments_clean)

print("\nCLINIC-WISE APPOINTMENT DISTRIBUTION")
print(master_df["clinic_name"].value_counts())

print("\nData integration completed successfully!")


DATA INTEGRATION REPORT
Appointment records: 1000
Clinic records: 8
Integrated records: 1000
Total columns: 24
Duplicate appointment IDs: 0
Unmatched clinics: 0

CLINIC-WISE APPOINTMENT DISTRIBUTION
clinic_name
Kothrud Family Clinic       162
Hadapsar Care Centre        145
Malviya Nagar Clinic        143
Edappally Medical Centre    142
Vaishali Nagar Clinic       130
Vijay Nagar Clinic          101
Gomti Nagar Care Point       92
Sitabuldi Health Point       85
Name: count, dtype: int64

Data integration completed successfully!


In [ ]:

# Step 4.10: Final Data Quality Validation

print("FINAL DATA QUALITY REPORT")
print("=" * 45)

print("Original appointments:", len(appointments))
print("Cleaned appointments:", len(master_df))
print("Duplicates removed:", len(appointments) - len(master_df))
print("Total columns:", master_df.shape[1])

# Automated quality checks
checks = {
    "Unique appointment IDs":
        master_df["appointment_id"].is_unique,

    "No duplicate rows":
        master_df.duplicated().sum() == 0,

    "Valid known ages":
        master_df["age"].dropna().between(0, 120).all(),

    "Valid known distances":
        master_df["distance_km"].dropna().ge(0).all(),

    "No negative lead times":
        master_df["lead_time_days"].dropna().ge(0).all(),

    "Standardized booking channels":
        master_df["booking_channel"].isin(
            ["App", "Phone", "Walk-in"]
        ).all(),

    "All clinics matched":
        master_df["clinic_name"].notna().all(),

    "All unique appointments retained":
        len(master_df) == len(appointments_clean)
}

print("\nAUTOMATED QUALITY CHECKS")
print("-" * 45)

for check, result in checks.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

print("\nPassed checks:", sum(checks.values()))
print("Total checks:", len(checks))

print("\nREMAINING MISSING VALUES")

missing = master_df.isna().sum()
print(missing[missing > 0])

print("\nFinal validation completed!")


FINAL DATA QUALITY REPORT
Original appointments: 1020
Cleaned appointments: 1000
Duplicates removed: 20
Total columns: 24

AUTOMATED QUALITY CHECKS
---------------------------------------------
Unique appointment IDs: PASS
No duplicate rows: PASS
Valid known ages: PASS
Valid known distances: PASS
No negative lead times: PASS
Standardized booking channels: PASS
All clinics matched: PASS
All unique appointments retained: PASS

Passed checks: 8
Total checks: 8

REMAINING MISSING VALUES
appt_date          6
age               16
distance_km       20
status             4
lead_time_days     6
dtype: int64

Final validation completed!


In [ ]:

# Step 4.11: Export Cleaned Datasets

import os

# Create output folder
output_dir = "/content/cleaned_data"

os.makedirs(output_dir, exist_ok=True)

# Verify quality checks before exporting
assert all(checks.values()), "Validation failed!"

# Export cleaned datasets
appointments_clean.to_csv(
    f"{output_dir}/appointments_clean.csv",
    index=False
)

clinics_clean.to_csv(
    f"{output_dir}/clinics_clean.csv",
    index=False
)

notes_clean.to_csv(
    f"{output_dir}/followup_notes_clean.csv",
    index=False
)

master_df.to_csv(
    f"{output_dir}/master_dataset.csv",
    index=False
)

# Export quality validation report
quality_report = pd.DataFrame({
    "Quality_Check": list(checks.keys()),
    "Result": [
        "PASS" if v else "FAIL"
        for v in checks.values()
    ]
})

quality_report.to_csv(
    f"{output_dir}/data_quality_report.csv",
    index=False
)

print("EXPORT REPORT")
print("=" * 40)

for file in sorted(os.listdir(output_dir)):
    print(file)

print("\nAll datasets exported successfully!")
print("Notebook 01 completed!")


EXPORT REPORT
appointments_clean.csv
clinics_clean.csv
data_quality_report.csv
followup_notes_clean.csv
master_dataset.csv

All datasets exported successfully!
Notebook 01 completed!


In [ ]:

# Step 5.1: Backup Cleaned Datasets

import shutil
from google.colab import files

shutil.make_archive(
    "/content/cleaned_data",
    "zip",
    "/content/cleaned_data"
)

files.download("/content/cleaned_data.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import os
print(os.listdir("/content"))

['.config', 'sample_data']


In [3]:
from google.colab import files

uploaded = files.upload()

Saving appointments.csv.csv to appointments.csv (1).csv
Saving clinics.csv.csv to clinics.csv.csv
Saving followup_notes.csv.csv to followup_notes.csv.csv


In [4]:
from pathlib import Path

data_dir = Path("/content/data")
data_dir.mkdir(exist_ok=True)

for name, content in uploaded.items():
    clean_name = name.replace(".csv.csv", ".csv")
    (data_dir / clean_name).write_bytes(content)

print("Original datasets ready:")
for file in data_dir.iterdir():
    print(file.name)

Original datasets ready:
appointments.csv (1).csv
clinics.csv
followup_notes.csv


In [5]:
from pathlib import Path
import pandas as pd

data_dir = Path("/content/data")

(data_dir / "appointments.csv (1).csv").rename(
    data_dir / "appointments.csv"
)

for name in ["appointments.csv", "clinics.csv",
             "followup_notes.csv"]:
    print(name, pd.read_csv(data_dir / name).shape)

appointments.csv (1020, 14)
clinics.csv (8, 6)
followup_notes.csv (30, 3)


In [6]:
import csv
import json

with open("/content/data/appointments.csv",
          newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    print("Header:", reader.fieldnames)
    print("Total rows:", len(rows))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020


In [7]:
def parse_row(row):
    result = row.copy()
    result["age"] = int(row["age"])
    result["fee_inr"] = int(row["fee_inr"])
    result["prior_no_shows"] = int(row["prior_no_shows"])
    result["distance_km"] = float(row["distance_km"])
    return result

In [8]:
accepted = []
rejected = []

In [9]:
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

Accepted: 0
Rejected: 0


In [11]:
accepted = []
rejected = []

for row in rows:
    try:
        converted = parse_row(row)
        accepted.append(converted)
    except ValueError:
        rejected.append(row)

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

Accepted: 988
Rejected: 32


In [12]:
with open("/content/rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

print("Task 1.1 completed successfully")

Task 1.1 completed successfully


In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/data/appointments.csv")

print("Original shape:", df.shape)
df.info()
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

Original shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
appointment_id      0
patient_id          0
clinic_id           0

In [14]:
df = df.drop_duplicates().copy()
print("After duplicates:", len(df))

df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

print(df["booking_channel"].unique())

After duplicates: 1000
['Phone' 'Walk-In' 'App']


In [15]:
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(df["booked_on"])

print("After date conversion:", len(df))

df = df.dropna(subset=["appt_date", "status"]).copy()

print("After cleaning:", len(df))
print("Final shape:", df.shape)

After date conversion: 1000
After cleaning: 990
Final shape: (990, 14)


In [17]:
invalid_age = (df["age"] < 1) | (df["age"] > 100)

print("Invalid ages:", invalid_age.sum())

df.loc[invalid_age, "age"] = np.nan

print("Missing ages:", df["age"].isna().sum())

age_median = df["age"].median()
distance_median = df["distance_km"].median()

df["age"] = df["age"].fillna(age_median).round().astype(int)

df["distance_km"] = df["distance_km"].fillna(distance_median)

print("Median age:", age_median)
print("Median distance:", distance_median)

Invalid ages: 5
Missing ages: 17
Median age: 48.0
Median distance: 8.25


In [18]:
# Task 1.3 - IQR outlier capping

for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    capped = ((df[col] < lower) |
              (df[col] > upper)).sum()

    df[col] = np.clip(df[col], lower, upper)

    print(col)
    print("Q1:", q1)
    print("Q3:", q3)
    print("Upper fence:", upper)
    print("Values capped:", capped)

df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Task 1.3 completed")

distance_km
Q1: 4.625
Q3: 13.0
Upper fence: 25.5625
Values capped: 26
fee_inr
Q1: 500.0
Q3: 787.5
Upper fence: 1218.75
Values capped: 3
Task 1.3 completed


In [19]:
# Task 1.4 - Feature engineering

df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

df["no_show"] = (
    df["status"] == "No-Show"
).astype(int)

X = df[["age", "distance_km",
        "fee_inr", "lead_days"]].to_numpy()

print("Total appointments:", len(df))
print("Total no-shows:", df["no_show"].sum())
print("No-show rate:", df["no_show"].mean() * 100)
print("Column means:", np.mean(X, axis=0))

mask = X[:, 3] > 14

print("Lead days > 14:", mask.sum())
print("Percentage:", mask.mean() * 100)

distance = X[:, 1]

z = (distance - np.mean(distance)) / np.std(distance)

print("Absolute z-score > 2:", (np.abs(z) > 2).sum())

print("Lead days range:",
      df["lead_days"].min(),
      df["lead_days"].max())

Total appointments: 990
Total no-shows: 218
No-show rate: 22.02020202020202
Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]
Lead days > 14: 296
Percentage: 29.898989898989896
Absolute z-score > 2: 53
Lead days range: 1 45


In [20]:
# Task 1.5 - Clinic API enrichment

import requests
import time

clinics = pd.read_csv("/content/data/clinics.csv")

temperatures = []

for _, clinic in clinics.iterrows():
    temp = None

    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": clinic["latitude"],
                    "longitude": clinic["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )

            response.raise_for_status()

            temp = response.json()["hourly"]["temperature_2m"][0]

            print(clinic["clinic_name"],
                  response.status_code, temp)
            break

        except (requests.exceptions.RequestException,
                KeyError, IndexError, ValueError) as e:
            print(clinic["clinic_name"], str(e))
            if attempt == 0:
                time.sleep(2)

    temperatures.append(temp)

clinics["temp_c"] = temperatures

clinics.to_csv(
    "/content/clinics_enriched.csv",
    index=False
)

print("Enriched clinic shape:", clinics.shape)

display(clinics)

Kothrud Family Clinic HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Kothrud Family Clinic HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Hadapsar Care Centre HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Hadapsar Care Centre HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Sitabuldi Health Point HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Sitabuldi Health Point HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Malviya Nagar Clinic HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Malviya Nagar Clinic HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Vaishali Nagar Clinic HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed

,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,None
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,None
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,None
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,None
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,None
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,None
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,None
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,None


In [21]:
# Task 1.6 - Clean dataset export

clean_df = df.sort_values("appointment_id").copy()

for col in ["booked_on", "appt_date"]:
    clean_df[col] = clean_df[col].dt.strftime("%Y-%m-%d")

columns = [
    "appointment_id", "patient_id", "clinic_id",
    "booked_on", "appt_date", "age", "gender",
    "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows",
    "status", "lead_days", "no_show"
]

clean_df = clean_df[columns]

print("Final shape:", clean_df.shape)
print("Unique IDs:", clean_df["appointment_id"].nunique())
print("Missing values:", clean_df.isna().sum().sum())

clean_df.to_csv(
    "/content/appointments_clean.csv",
    index=False
)

print("CSV exported successfully!")

Final shape: (990, 16)
Unique IDs: 990
Missing values: 0
CSV exported successfully!


In [22]:
# Task 1.6 - Generate SQL load script

def sql_value(value):
    if pd.isna(value):
        return "NULL"
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"
    return str(value)

statements = []

for row in clinics.itertuples(index=False, name=None):
    values = ", ".join(sql_value(v) for v in row)
    statements.append(
        f"INSERT INTO clinics VALUES ({values});"
    )

for row in clean_df.itertuples(index=False, name=None):
    values = ", ".join(sql_value(v) for v in row)
    statements.append(
        f"INSERT INTO appointments VALUES ({values});"
    )

with open("/content/load_data.sql", "w") as f:
    f.write("\n".join(statements))

print("Clinic records:", len(clinics))
print("Appointment records:", len(clean_df))
print("Total INSERT statements:", len(statements))
print("SQL file generated successfully!")

Clinic records: 8
Appointment records: 990
Total INSERT statements: 998
SQL file generated successfully!


In [23]:
from google.colab import files
import zipfile

required_files = [
    "rejected_rows.json",
    "appointments_clean.csv",
    "clinics_enriched.csv",
    "load_data.sql",
    "data/appointments.csv",
    "data/clinics.csv",
    "data/followup_notes.csv"
]

with zipfile.ZipFile(
    "/content/part1_final.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as z:
    for name in required_files:
        z.write("/content/" + name, name)

print("Part 1 backup ready!")

files.download("/content/part1_final.zip")

Part 1 backup ready!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
# Task 2.2 - Python cross-check

check = clean_df.groupby("clinic_id").agg(
    appointments=("appointment_id", "count"),
    no_shows=("no_show", "sum")
).reset_index()

check = check.merge(
    clinics[["clinic_id", "clinic_name", "city"]],
    on="clinic_id"
)

check["no_show_pct"] = (
    100 * check["no_shows"] / check["appointments"]
).round(1)

display(check.sort_values(
    "no_show_pct", ascending=False
))

,clinic_id,appointments,no_shows,clinic_name,city,no_show_pct
7,C08,90,30,Gomti Nagar Care Point,Lucknow,33.3
1,C02,143,39,Hadapsar Care Centre,Pune,27.3
3,C04,142,38,Malviya Nagar Clinic,Jaipur,26.8
0,C01,160,36,Kothrud Family Clinic,Pune,22.5
6,C07,101,21,Vijay Nagar Clinic,Indore,20.8
5,C06,142,23,Edappally Medical Centre,Kochi,16.2
2,C03,82,13,Sitabuldi Health Point,Nagpur,15.9
4,C05,130,18,Vaishali Nagar Clinic,Jaipur,13.8
